# DL_DOA — Baseline vs. Physics-Informed Architecture: Head-to-Head Comparison

Ei notebook duita jinis compare kore:

1. **Main architecture (baseline)** — paper-er original pretrained **UNet**, exactly jemon ache repo-te.
2. **New architecture (PIA-Net)** — ekta notun **Physics-Informed Attention Network**, jeta:
   - raw antenna observation-ke shorashori **known array-physics dictionary**-r upor project kore
     (kono naive nearest-neighbor upsampling nei),
   - ekta **learned-ISTA (unfolded sparse recovery)** block diye angle-domain-e sparse map বের kore,
   - ekta **self-attention** block diye kache-kachi/weak source resolve korte shahajjo kore,
   - tarpor same 256x256 output-e decode kore, jate **same evaluation code** diye dutake fair-vabe
     compare kora jay.

**Comparison-e ja ja thakbe:** parameter count, inference latency (computation cost), RMSE vs SNR,
Pd (probability of detection) vs SNR, ebong SNR-er upor performance kototuku drop kore (robustness)।

Ei notebook-ta **onek gulo choto cell-e bhaga**, protiti step-e visualization ache — jate step-by-step
bujha jay kano notun architecture-ta emon design kora holo, ar prottekta piece ki kore.

**Important honesty note:** Baseline UNet paper-er full Table-I setup-e (500 epoch, 10000 sample/epoch)
train kora — eta already pretrained hishebe repo-te ache, tai amra shudhu load kori (real, published-quality
number). PIA-Net eituntuk notebook-e-i freshly train kora hoy, onek kom epoch diye (Colab-e shomoy shimito) —
tai eta ekটা **completely fair fight na**, ekta *first-look* comparison। Fair comparison-er jonno PIA-Net-ke
o similar epoch/steps diye train korte hobe (niche `EPOCHS` variable change kore).

## Part 0 — Setup

In [ ]:
!git clone https://github.com/Mishatmilon059/DL_DOA_CLONE.git
%cd DL_DOA_CLONE

In [ ]:
!pip install -q tensorflow==2.21.0 numpy==2.4.6 scipy==1.17.1 matplotlib==3.11.1 \
    scikit-learn==1.9.0 tqdm==4.70.0 opencv-python-headless==5.0.0

In [ ]:
!apt-get -qq install -y p7zip-full > /dev/null
%cd DL_DOA/models
!7z x -y inf_model_007_256_unet.7z.001 > /dev/null
%cd /content/DL_DOA_CLONE/DL_DOA

In [ ]:
import os, sys, time, itertools
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.append('.')

import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras import layers as L

from src.tvt_data_generation_v3 import (
    ev, beamforming_vector_generation_P, beamforming_vector_generation_Q,
    generate_points, generate_channel_v2, generate_noise, generate_gt,
    get_real_imag, myarray, validation_data_generator,
)
from src.tvt_models import UNet
from src.TVT_Blob_Inference import (
    get_blob_detector, get_blob_peaks, peaks_to_angles,
    prepare_for_metric, get_ang_difference, filter_angles,
)

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

**Fixed regime for this whole notebook**: `nt = nr = P = Q = 16` — eta paper-er nijer Fig. 5-6
test condition (L=3, SNR sweep, QP=16). Physics dictionary-ta ei ekta fixed antenna/codebook config-er
jonno build kora hocche, tai training o evaluation dutoi eki regime-e thakbe (fair comparison-er jonno
dorkar).

In [ ]:
NT = NR = P_CB = Q_CB = 16   # antennas & codebook size (paper Fig. 5-6 condition)
SIGMA = 0.07
M_OUT = 256                  # final output grid (same as baseline)
G_GRID = 32                  # PIA-Net's internal angle-grid resolution
print(f'nt=nr={NT}, P=Q={P_CB}, output grid={M_OUT}x{M_OUT}, PIA-Net angle grid={G_GRID}x{G_GRID}')

## Part 1 — Paper-er physical model-ta ki bole, dekhe bujhi

### Part 1.1 — Steering vector `a(angle)` ki dekhte

Antenna array-er proti direction-e ekta "signature" (complex vector) ache — eita-i steering vector.
Magnitude shobsomoy shomo (1/sqrt(nt)), kintu **phase** antenna-theke-antenna linearly barhe — eita-i
angle-er information carry kore।

In [ ]:
sample_angle = np.deg2rad(50)
a = ev(NT, sample_angle).flatten()

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].stem(np.abs(a))
axs[0].set_title('Magnitude |a(angle)| — shob antenna-e shoman')
axs[0].set_xlabel('Antenna index')
axs[1].plot(np.unwrap(np.angle(a)), marker='o')
axs[1].set_title(f'Phase of a(angle) at angle={np.rad2deg(sample_angle):.0f}°\n(linear ramp = angle information)')
axs[1].set_xlabel('Antenna index')
plt.tight_layout(); plt.show()

### Part 1.2 — Beamforming codebook F, W (DFT-based, paper Eq. 16)

In [ ]:
F = beamforming_vector_generation_P(P_CB, NT)
W = beamforming_vector_generation_Q(Q_CB, NR)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
im0 = axs[0].imshow(np.abs(F), cmap='viridis'); axs[0].set_title('|F| (TX codebook)')
im1 = axs[1].imshow(np.angle(F), cmap='twilight'); axs[1].set_title('phase(F)')
for ax in axs: ax.set_xlabel('beam index'); ax.set_ylabel('antenna index')
plt.tight_layout(); plt.show()

### Part 1.3 — Ekta example generate kori: raw (un-upsampled) observation

Ei function-ta repo-r nijer building block gulo (`generate_channel_v2`, `generate_noise`, `generate_gt`)
use kore, kintu **zoom/upsample step baad diye** — tai output shorashori raw `Q x P` complex matrix,
64x64-e naive-vabe stretch kora na।

In [ ]:
def generate_raw_example(L, SNR, rng, sigma=SIGMA, M=M_OUT):
    alpha_l = np.sqrt(1/L) * (rng.standard_normal(L) + 1j*rng.standard_normal(L)) / np.sqrt(2)
    s_idx = np.argsort(np.abs(alpha_l)); alpha_l = alpha_l[s_idx[::-1]]
    points = generate_points(L, np.pi/6, rng=rng)
    phi_l = [p[0] for p in points]; psi_l = [p[1] for p in points]
    angle_v = np.hstack([phi_l, psi_l])
    omega_phi = np.pi*np.cos(phi_l); omega_psi = -np.pi*np.cos(psi_l)

    H = generate_channel_v2(NR, NT, angle_v, alpha_l)
    Gmat = (W.view(myarray).H @ H) @ F
    Z = generate_noise(1.0, SNR, P_CB, Q_CB, rng=rng)
    Y = Gmat + Z                                   # raw Q x P complex observation

    gt = generate_gt(L, np.ones(L), omega_phi, omega_psi, num_points_rx=M, num_points_tx=M, sigma=sigma)
    feat = np.stack([psi_l, phi_l])                # true (psi, phi) in radians
    return Y, np.expand_dims(gt, -1).astype(np.float32), feat.astype(np.float32)

rng_demo = np.random.default_rng(7)
Y_demo, gt_demo, feat_demo = generate_raw_example(L=3, SNR=10, rng=rng_demo)
print('Raw Y shape:', Y_demo.shape, Y_demo.dtype)
print('GT heatmap shape:', gt_demo.shape)
print('True (psi, phi) [rad]:\n', feat_demo)

### Part 1.4 — Raw observation vs. baseline's naive-upsampled input, side by side

In [ ]:
zoom_factor = 4
data_up = np.dstack([
    __import__('scipy.ndimage', fromlist=['zoom']).zoom(Y_demo.real, zoom_factor, order=0),
    __import__('scipy.ndimage', fromlist=['zoom']).zoom(Y_demo.imag, zoom_factor, order=0),
])

fig, axs = plt.subplots(1, 4, figsize=(15, 3.5))
axs[0].imshow(Y_demo.real, cmap='RdBu'); axs[0].set_title(f'Raw Re(Y) — {P_CB}x{P_CB}\n(what PIA-Net sees)')
axs[1].imshow(Y_demo.imag, cmap='RdBu'); axs[1].set_title(f'Raw Im(Y) — {P_CB}x{P_CB}')
axs[2].imshow(data_up[:, :, 0], cmap='RdBu'); axs[2].set_title('Nearest-neighbor upsampled\n64x64 (what baseline sees)')
axs[3].imshow(gt_demo[:, :, 0], cmap='hot'); axs[3].set_title('Ground truth (256x256)')
for ax in axs: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print('Lokkho koro: cell [2]-er blocky pattern — eta raw data-r kono notun information na,')
print('shudhu each raw pixel-ke 4x4 repeat kora hoyeche. Baseline network-ke ei blocky, meaningless')
print('pattern theke nijei kaj korte hoy. PIA-Net eta korbe na — eta raw cell [0]/[1] shorashori nibe.')

## Part 2 — Main architecture (baseline): pretrained UNet load koro

In [ ]:
baseline_model = UNet(M=M_OUT)
baseline_model.load_weights('models/inf_model_007_256_unet.h5')
baseline_params = baseline_model.count_params()
print(f'Baseline UNet loaded. Total parameters: {baseline_params:,}')

### Part 2.1 — Baseline-er prediction dekhi Part 1-er example-e

In [ ]:
import scipy.ndimage as ndi
data_up_f32 = data_up.astype(np.float32)
pred_baseline_demo = baseline_model(tf.expand_dims(data_up_f32, axis=0), training=False)
pred_baseline_demo = tf.squeeze(pred_baseline_demo, axis=0)

detector = get_blob_detector()
peaks_b, amps_b = get_blob_peaks(pred_baseline_demo, detector)
order = np.argsort(-amps_b)
peaks_b = peaks_b[order[:3]] if len(peaks_b) else peaks_b

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].imshow(gt_demo[:, :, 0], cmap='hot'); axs[0].set_title('Ground truth')
axs[1].imshow(pred_baseline_demo.numpy()[:, :, 0], cmap='hot'); axs[1].set_title('Baseline UNet prediction')
if len(peaks_b): axs[1].scatter(peaks_b[:, 0], peaks_b[:, 1], c='cyan', marker='x', s=70, label='detected peaks')
axs[1].legend()
for ax in axs: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Part 3 — PIA-Net-er core idea: physics dictionary

Ekta grid-er upor shob shombhabbo (phi, psi) angle-pair-er jonno — "eikhane ekta source thakle observation
dekhte kemon hoto" seta age-theke-i jana jay (Part 1.1-2-er formula theke)। Eita-i "dictionary" — ekta
lookup jeta directly array-physics theke ashe, kono data theke shekha na।

**Key math trick** (jachai kora hoyeche numpy-te finite-difference diye, error ~1e-9): eita dictionary
ta shomporke ekta boro `(Q*P) x (G*G)` matrix banano lagbe na — eita duita CHOTO matrix `U` (Q x G) ar
`V` (P x G)-e bhenge fela jay, karon:

&nbsp;&nbsp;&nbsp;&nbsp;`observation_from_angle(i,j) = c * outer(U[:,i], V[:,j])`

Eita onek fast ar memory-efficient।

In [ ]:
psis = np.linspace(0.05, np.pi - 0.05, G_GRID)
phis = np.linspace(0.05, np.pi - 0.05, G_GRID)

A_r = np.hstack([ev(NR, a) for a in psis])   # nr x G  (AoA steering vectors on the grid)
A_t = np.hstack([ev(NT, a) for a in phis])   # nt x G  (AoD steering vectors on the grid)

U_dict = (W.conj().T @ A_r).astype(np.complex64)   # Q x G
V_dict = (F.T @ A_t.conj()).astype(np.complex64)   # P x G
DICT_C = np.float32(np.sqrt(NT * NR))

print('U_dict shape:', U_dict.shape, ' V_dict shape:', V_dict.shape)
print(f'(Compare: a full non-separable dictionary would be ({P_CB*Q_CB}, {G_GRID*G_GRID}) — '
      f'{P_CB*Q_CB*G_GRID*G_GRID:,} complex numbers. We only store '
      f'{U_dict.size + V_dict.size:,}.)')

### Part 3.1 — Dictionary-er kichu individual 'atom' dekhi

In [ ]:
def dict_atom(i, j):
    return DICT_C * np.outer(U_dict[:, i], V_dict[:, j])

fig, axs = plt.subplots(2, 4, figsize=(14, 6))
demo_ij = [(4, 4), (4, 28), (16, 16), (28, 4)]
for col, (i, j) in enumerate(demo_ij):
    atom = dict_atom(i, j)
    axs[0, col].imshow(np.abs(atom), cmap='viridis')
    axs[0, col].set_title(f'|atom| at\npsi={np.rad2deg(psis[i]):.0f}°, phi={np.rad2deg(phis[j]):.0f}°')
    axs[1, col].imshow(np.angle(atom), cmap='twilight')
    axs[1, col].set_title('phase')
    for r in range(2): axs[r, col].set_xticks([]); axs[r, col].set_yticks([])
plt.suptitle('Each atom = "what the raw observation would look like if a single source sat exactly here"')
plt.tight_layout(); plt.show()

## Part 4 — Hand-e (pure numpy) ekta sparse-recovery iteration dekhi

Idea shohoj: shuru kori empty angle-map (shob shunno) diye। Protiti step-e:

1. Amader current guess theke observation "reconstruct" kori (forward model),
2. Real observation-er shathe compare kore ki baki ache (residual) dekhi,
3. Shei residual-ke abar angle-domain-e "back-project" kori (adjoint),
4. Choto step nei, ar weak/noise-moto value-gulo 0-e clip kore dei (soft-threshold) — eita-i sparse
   thakar guarantee dey (jehetu amra jani mathe গুটিকয়েক source-i ache, hundreds na)।

Eta 4-5 bar repeat korle sparse map dhire dhire true angle-gulor kache converge kore।

In [ ]:
def ista_forward(X):
    return DICT_C * (U_dict @ X.astype(np.complex64) @ V_dict.T)

def ista_adjoint(R):
    return DICT_C * np.real(U_dict.conj().T @ R @ V_dict.conj())

def run_ista_numpy(Y, n_iters=6, step=0.06, thresh=0.015):
    X = np.zeros((G_GRID, G_GRID), dtype=np.float32)
    history = [X.copy()]
    for k in range(n_iters):
        R = Y - ista_forward(X)
        grad = ista_adjoint(R)
        X = np.maximum(X + step * grad - thresh, 0)
        history.append(X.copy())
    return history

history = run_ista_numpy(Y_demo)
print(f'Ran {len(history)-1} ISTA iterations on the Part-1 example.')

In [ ]:
true_i = [np.argmin(np.abs(psis - p)) for p in feat_demo[0]]
true_j = [np.argmin(np.abs(phis - p)) for p in feat_demo[1]]

fig, axs = plt.subplots(1, len(history), figsize=(3 * len(history), 3.2))
for k, X in enumerate(history):
    axs[k].imshow(X.T, origin='lower', cmap='hot', extent=[0, G_GRID, 0, G_GRID])
    axs[k].scatter(true_i, true_j, facecolors='none', edgecolors='cyan', s=90, linewidths=1.5)
    axs[k].set_title('start' if k == 0 else f'iter {k}')
    axs[k].set_xticks([]); axs[k].set_yticks([])
plt.suptitle('Sparse angle-map convergence (cyan circles = true angle locations)')
plt.tight_layout(); plt.show()

print('Dekho: shuru-te shob 0, protiti iteration-e bright spot-gulo cyan circle-er dike agiye ashe.')
print('Eita-i "unfolded" sparse recovery — protita iteration ekta network layer hobe (Part 5), ')
print('ar step/threshold parameter-gulo train kore aro better kora jabe (Learned-ISTA / LISTA).')

## Part 5 — Ei ISTA iteration-ke trainable Keras layer banai (Learned-ISTA)

Part 4-e step-size ar threshold hand-set kora hoyechilo. Ekhon egulo-ke **learnable parameter** banabo —
training-er shomoy network nijei best step/threshold khunje nebe protiti iteration-er jonno. Ei idea-ta
2024-25 shaler kayekta published paper-e ache (IRLS-NET, MoD-DNN — Field Guide artifact-e reference dewa
ache) — notun na, kintu ei exact problem-e kew apply kore nai.

In [ ]:
class LearnedISTA(tf.keras.layers.Layer):
    def __init__(self, U_dict, V_dict, dict_c, n_iters=4, **kwargs):
        super().__init__(**kwargs)
        self.U = tf.constant(U_dict, dtype=tf.complex64)
        self.V = tf.constant(V_dict, dtype=tf.complex64)
        self.c = tf.constant(dict_c, dtype=tf.float32)
        self.c_complex = tf.complex(self.c, tf.constant(0.0, dtype=tf.float32))
        self.n_iters = n_iters
        self.G = U_dict.shape[1]

    def build(self, input_shape):
        self.steps = [self.add_weight(name=f'step_{k}', shape=(), dtype=tf.float32,
                                       initializer=tf.keras.initializers.Constant(0.06))
                      for k in range(self.n_iters)]
        self.thresholds = [self.add_weight(name=f'thresh_{k}', shape=(), dtype=tf.float32,
                                            initializer=tf.keras.initializers.Constant(0.015))
                            for k in range(self.n_iters)]

    def call(self, y_real_imag):
        # y_real_imag: (batch, Q, P, 2) raw complex observation as 2 real channels
        Y = tf.complex(y_real_imag[..., 0], y_real_imag[..., 1])
        batch = tf.shape(Y)[0]
        X = tf.zeros((batch, self.G, self.G), dtype=tf.float32)
        Uc = tf.math.conj(self.U)
        Vc = tf.math.conj(self.V)
        for k in range(self.n_iters):
            Xc = tf.cast(X, tf.complex64)
            Yhat = self.c_complex * tf.einsum('qi,bij,pj->bqp', self.U, Xc, self.V)
            R = Y - Yhat
            grad = self.c * tf.math.real(tf.einsum('qi,bqp,pj->bij', Uc, R, Vc))
            X = tf.nn.relu(X + self.steps[k] * grad - self.thresholds[k])
        return tf.expand_dims(X, axis=-1)   # (batch, G, G, 1)

In [ ]:
# Sanity check: run one un-trained forward pass, shape ঠিক ache kina dekho
lista_test = LearnedISTA(U_dict, V_dict, DICT_C, n_iters=4)
test_input = tf.expand_dims(np.dstack([Y_demo.real, Y_demo.imag]).astype(np.float32), axis=0)
test_out = lista_test(test_input)
print('LearnedISTA output shape:', test_out.shape, '(expect: (1,', G_GRID, ',', G_GRID, ', 1))')

## Part 6 — Self-attention refinement block

ISTA-r por sparse map-e prottekta grid-cell ekla-i kaj kore — kintu bastobe kache-kachi angle-gulor
modhye interaction thaka uchit (ekta strong source pashe thakle weak source dhaka pore jete pare)।
Ekta choto **self-attention** layer diye grid-er shob cell-ke ekbar "ek-shathe kotha bolar" shujog dei —
eita-i TransMUSIC/SubspaceNet-er moto paper-e low-SNR ar kache-kachi source resolve korar jonno use hoy।

In [ ]:
def attention_refine_block(x, d_model=16, n_heads=2):
    shape = x.shape[1:3]
    h = L.Conv2D(d_model, 1, padding='same')(x)
    seq = L.Reshape((shape[0] * shape[1], d_model))(h)
    attn_out = L.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)(seq, seq)
    seq = L.Add()([seq, attn_out])
    seq = L.LayerNormalization()(seq)
    h = L.Reshape((shape[0], shape[1], d_model))(seq)
    h = L.Conv2D(1, 1, padding='same', activation='relu')(h)
    return L.Add()([x, h])

print('attention_refine_block defined — ekta residual self-attention block '
      f'jeta {G_GRID}x{G_GRID} sparse map-er upor kaj kore.')

## Part 7 — Shob piece jog kore PIA-Net toiri kori

Pipeline: raw (16,16,2) → LearnedISTA (angle-domain sparse map, GxG) → attention refine → choto decoder
→ (256,256,1) output। Output shape/interpretation baseline-er shathe **identical** — tai same loss ar
same evaluation code use kora jabe।

In [ ]:
def build_pia_net(g_grid=G_GRID, m_out=M_OUT, n_ista_iters=4):
    inputs = tf.keras.Input(shape=(P_CB, Q_CB, 2), name='raw_observation')
    x = LearnedISTA(U_dict, V_dict, DICT_C, n_iters=n_ista_iters, name='learned_ista')(inputs)
    x = attention_refine_block(x, d_model=16, n_heads=2)

    upsample_steps = int(np.log2(m_out // g_grid))
    filt = 32
    for _ in range(upsample_steps):
        x = L.Conv2DTranspose(filt, 3, strides=2, padding='same')(x)
        x = L.BatchNormalization()(x)
        x = L.Activation('relu')(x)
        filt = max(filt // 2, 8)
    outputs = L.Conv2D(1, 3, padding='same', activation='linear', name='heatmap')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='PIA-Net')

pia_net = build_pia_net()
pia_net.summary()

In [ ]:
pia_params = pia_net.count_params()
print(f'PIA-Net total parameters: {pia_params:,}')
print(f'Baseline UNet parameters: {baseline_params:,}')
ratio = baseline_params / pia_params
print(f'\nBaseline has {ratio:.1f}x {"more" if ratio > 1 else "fewer"} parameters than PIA-Net.')

## Part 8 — Computation-cost comparison (architecture-level, training-er age-i measure kora jay)

In [ ]:
def benchmark_latency(model, input_shape, n_runs=30):
    dummy = tf.random.normal((1,) + input_shape)
    _ = model(dummy, training=False)  # warm-up (graph tracing)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = model(dummy, training=False)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.std(times)

lat_baseline_mean, lat_baseline_std = benchmark_latency(baseline_model, (64, 64, 2))
lat_pia_mean, lat_pia_std = benchmark_latency(pia_net, (P_CB, Q_CB, 2))

print(f'Baseline UNet : {lat_baseline_mean:.2f} ± {lat_baseline_std:.2f} ms / example')
print(f'PIA-Net       : {lat_pia_mean:.2f} ± {lat_pia_std:.2f} ms / example')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
names = ['Baseline\nUNet', 'PIA-Net']

axs[0].bar(names, [baseline_params, pia_params], color=['#5a6472', '#c4460f'])
axs[0].set_title('Parameter count')
axs[0].set_yscale('log')
for i, v in enumerate([baseline_params, pia_params]):
    axs[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

axs[1].bar(names, [lat_baseline_mean, lat_pia_mean],
           yerr=[lat_baseline_std, lat_pia_std], capsize=6, color=['#5a6472', '#c4460f'])
axs[1].set_title('Inference latency (ms/example, batch=1)')

plt.tight_layout(); plt.show()

## Part 9 — PIA-Net train korar jonno data pipeline (paper-er nijer generator)

Paper-er Table I-er distribution-i use korchi (L random 1-9, SNR random [-15,24] dB) — kintu **P=Q=nt=nr=16
fixed** rakhchi (Part 0-er note onujayi, dictionary-r shathe match korte)। Eita-i "original data from the
paper" — kono outside/user dataset na, shorashori repo-r nijer physics-based generator theke।

In [ ]:
def raw_training_generator():
    rng = np.random.default_rng()
    while True:
        L_paths = int(rng.integers(1, 10))
        SNR = int(rng.integers(-15, 25))
        Y, gt, _ = generate_raw_example(L_paths, SNR, rng)
        data = np.dstack([Y.real, Y.imag]).astype(np.float32)
        yield data, gt

def raw_val_generator(conditions, examples_per_condition=1, seed=42):
    rng = np.random.default_rng(seed)
    for (L_paths, SNR) in conditions:
        for _ in range(examples_per_condition):
            Y, gt, feat = generate_raw_example(L_paths, SNR, rng)
            data = np.dstack([Y.real, Y.imag]).astype(np.float32)
            yield data, gt, feat, np.array([L_paths, SNR], dtype=np.float32)

train_ds = tf.data.Dataset.from_generator(
    raw_training_generator,
    output_signature=(
        tf.TensorSpec(shape=(P_CB, Q_CB, 2), dtype=tf.float32),
        tf.TensorSpec(shape=(M_OUT, M_OUT, 1), dtype=tf.float32),
    ),
).batch(16).prefetch(tf.data.AUTOTUNE)

val_conditions_train = [(int(l), int(s)) for l, s in zip(
    np.random.default_rng(123).integers(1, 10, 100),
    np.random.default_rng(124).integers(-15, 25, 100))]
val_ds_train = tf.data.Dataset.from_generator(
    lambda: ((d, g) for d, g, f, m in raw_val_generator(val_conditions_train, 1, seed=42)),
    output_signature=(
        tf.TensorSpec(shape=(P_CB, Q_CB, 2), dtype=tf.float32),
        tf.TensorSpec(shape=(M_OUT, M_OUT, 1), dtype=tf.float32),
    ),
).batch(16).prefetch(tf.data.AUTOTUNE)

print('Training + validation tf.data pipelines ready.')

## Part 10 — PIA-Net train koro

### Part 10.1 — Ekta gotcha: plain MSE ei sparse target-e "shob shunno predict koro" e atke jay

Ground-truth heatmap-er beshirbhag pixel-i 0 — matro L-ta choto Gaussian bump chhara। Tai plain MSE loss
diye train korle network khub shohoje ekta "lazy" local minimum-e atke jay: **shob jaygay 0 (ba kachakachi
0) predict kora**, jeta already onek kom loss dey (jehetu target-er 99%+ pixel-o 0)। Eta amra nijei test
kore dekhechi: matro kayekta epoch-er por prediction-er std ~0.01 (proyoshoi flat), r blob-detector kono
peak-i khunje pay na।

**Fix**: ekta **weighted MSE** — jekhane ground-truth beshi (mane real source ache), okhane loss-ke beshi
weight dei, jate network shei small-but-important region-gulo-r upor beshi mnoyog dey। Eita heatmap-based
keypoint-detection literature-e ekta standard trick।

In [ ]:
def weighted_mse(alpha=8.0):
    """Loss weight = 1 + alpha*gt -- background (gt=0) pixels keep weight 1,
    foreground (gt>0, i.e. near a true angle) pixels get up-weighted so the
    network can't just coast by predicting all-zero."""
    def loss_fn(y_true, y_pred):
        w = 1.0 + alpha * y_true
        return tf.reduce_mean(w * tf.square(y_pred - y_true))
    return loss_fn

print('weighted_mse defined. Compare: plain tf.keras.losses.MeanSquaredError() would collapse here.')

`EPOCHS`/`STEPS_PER_EPOCH` choto rakha ache (druto demo-r jonno) — paper-er Table I-er full setup
`epochs=500, steps_per_epoch=312` (10000 sample/epoch, batch 32)। Fair comparison-er jonno eta barhate
paro, kintu Colab-e onek shomoy nebe. Learning rate-o (`0.003`) plain baseline-er (`0.001`) theke beshi
rakha hoyeche, karon PIA-Net onek chotto (~7.5K param) ar druto converge korte pare.

In [ ]:
EPOCHS = 15              # paper: 500
STEPS_PER_EPOCH = 80     # paper: 312
VAL_STEPS = 6

pia_net.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss=weighted_mse(alpha=8.0))

history = pia_net.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds_train,
    validation_steps=VAL_STEPS,
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train loss', marker='o')
plt.plot(history.history['val_loss'], label='val loss', marker='s')
plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('PIA-Net training curve')
plt.legend(); plt.grid(True); plt.show()

## Part 11 — Baseline vs PIA-Net: SNR-er upor several example dekhi (visual comparison)

In [ ]:
def predict_both(L_paths, SNR, rng):
    Y, gt, feat = generate_raw_example(L_paths, SNR, rng)
    raw_in = np.dstack([Y.real, Y.imag]).astype(np.float32)
    zoomed_in = np.dstack([
        ndi.zoom(Y.real, zoom_factor, order=0),
        ndi.zoom(Y.imag, zoom_factor, order=0),
    ]).astype(np.float32)

    pred_b = tf.squeeze(baseline_model(tf.expand_dims(zoomed_in, 0), training=False), 0)
    pred_p = tf.squeeze(pia_net(tf.expand_dims(raw_in, 0), training=False), 0)
    return gt, pred_b, pred_p, feat

showcase_snrs = [-10, 0, 20]
fig, axs = plt.subplots(len(showcase_snrs), 3, figsize=(10, 3.3 * len(showcase_snrs)))
rng_show = np.random.default_rng(2026)
for row, snr in enumerate(showcase_snrs):
    gt, pred_b, pred_p, feat = predict_both(3, snr, rng_show)
    axs[row, 0].imshow(gt[:, :, 0], cmap='hot'); axs[row, 0].set_ylabel(f'SNR={snr}dB', fontsize=11)
    axs[row, 1].imshow(pred_b.numpy()[:, :, 0], cmap='hot')
    axs[row, 2].imshow(pred_p.numpy()[:, :, 0], cmap='hot')
    if row == 0:
        axs[row, 0].set_title('Ground truth')
        axs[row, 1].set_title('Baseline UNet')
        axs[row, 2].set_title('PIA-Net')
    for c in range(3): axs[row, c].set_xticks([]); axs[row, c].set_yticks([])
plt.tight_layout(); plt.show()

## Part 12 — Pura evaluation: dutokei paper-er exact Fig. 5-6 condition-e test kori

`L=3`, `SNR = -10..25 (step 5)`, `QP=16` — ei same condition-e, same blob-detection + Hungarian-matching
metric code (repo-r `src/TVT_Blob_Inference.py` theke import kora) use kore duto model-i evaluate kora
hocche — tai comparison-ta apples-to-apples.

In [ ]:
N_PER_CONDITION = 200   # paper: 1000. Druto test-er jonno choto; None dile beshi shomoy lagbe.
SNR_VALUES = list(range(-10, 30, 5))

def evaluate_model(predict_fn, model_name):
    results = {}
    rng = np.random.default_rng(42)
    for snr in SNR_VALUES:
        for _ in range(N_PER_CONDITION):
            Y, gt, feat = generate_raw_example(3, snr, rng)
            pred = predict_fn(Y)
            peaks, amps = get_blob_peaks(pred, detector)
            order = np.argsort(-amps)
            peaks = peaks[order[:3]] if len(peaks) else peaks
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M_OUT)
            gt_a, pred_a = prepare_for_metric(angles_est, feat)
            results.setdefault(snr, []).append({'gt': gt_a, 'pred': pred_a})

    # Same good/bad accounting as src/TVT_Blob_Inference.py: each example contributes
    # 2*L difference values (psi errors AND phi errors, flattened together), so Pd's
    # denominator must be len(good)+len(bad) over ALL those entries -- not just L per
    # example, which would silently double-count and let Pd exceed 1.
    rmse, pd = {}, {}
    for snr, examples in results.items():
        good_all, bad_all = [], []
        for ex in examples:
            if np.isnan(ex['pred']).any():
                bad_all.append(np.full(ex['gt'].size, 999.0))  # count as fully missed
                continue
            diffs = get_ang_difference(ex['gt'], ex['pred'])
            good, bad = filter_angles(diffs, max_deg_error=1.0)
            good_all.append(good)
            bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) else np.nan
        pd[snr] = len(good_all) / total if total else np.nan
    return rmse, pd

def predict_baseline(Y):
    zoomed = np.dstack([ndi.zoom(Y.real, zoom_factor, order=0), ndi.zoom(Y.imag, zoom_factor, order=0)]).astype(np.float32)
    return tf.squeeze(baseline_model(tf.expand_dims(zoomed, 0), training=False), 0)

def predict_pia(Y):
    raw_in = np.dstack([Y.real, Y.imag]).astype(np.float32)
    return tf.squeeze(pia_net(tf.expand_dims(raw_in, 0), training=False), 0)

print('Evaluating baseline UNet ...')
rmse_baseline, pd_baseline = evaluate_model(predict_baseline, 'baseline')
print('Evaluating PIA-Net ...')
rmse_pia, pd_pia = evaluate_model(predict_pia, 'pia')
print('Done.')

## Part 13 — RMSE / Pd comparison plots

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
snrs_sorted = sorted(rmse_baseline.keys())

axs[0].plot(snrs_sorted, [rmse_baseline[s] for s in snrs_sorted], marker='o', label='Baseline UNet', color='#5a6472')
axs[0].plot(snrs_sorted, [rmse_pia[s] for s in snrs_sorted], marker='o', label='PIA-Net', color='#c4460f')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (degrees)'); axs[0].set_title('RMSE vs SNR (L=3, QP=16)')
axs[0].grid(True); axs[0].legend()

axs[1].plot(snrs_sorted, [pd_baseline[s] for s in snrs_sorted], marker='s', label='Baseline UNet', color='#5a6472')
axs[1].plot(snrs_sorted, [pd_pia[s] for s in snrs_sorted], marker='s', label='PIA-Net', color='#c4460f')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_ylim(0, 1.05)
axs[1].set_title('Detection probability vs SNR'); axs[1].grid(True); axs[1].legend()

plt.tight_layout(); plt.show()

### Part 13.1 — Robustness: worst-case (lowest SNR) performance, ar SNR-drop-e koto kome

In [ ]:
low_snr, high_snr = snrs_sorted[0], snrs_sorted[-1]

def robustness_report(name, rmse_d, pd_d):
    drop = rmse_d[low_snr] - rmse_d[high_snr]
    print(f'{name}:')
    print(f'  RMSE @ {low_snr}dB (worst case) = {rmse_d[low_snr]:.3f}°   |  Pd @ {low_snr}dB = {pd_d[low_snr]:.3f}')
    print(f'  RMSE @ {high_snr}dB (best case)  = {rmse_d[high_snr]:.3f}°   |  Pd @ {high_snr}dB = {pd_d[high_snr]:.3f}')
    print(f'  RMSE degrades by {drop:.3f}° going from {high_snr}dB to {low_snr}dB\n')

robustness_report('Baseline UNet', rmse_baseline, pd_baseline)
robustness_report('PIA-Net', rmse_pia, pd_pia)

## Part 14 — Final summary

In [ ]:
print(f'{"Metric":<32}{"Baseline UNet":<18}{"PIA-Net":<18}')
print('-' * 68)
print(f'{"Parameters":<32}{baseline_params:<18,}{pia_params:<18,}')
print(f'{"Latency (ms/example)":<32}{lat_baseline_mean:<18.2f}{lat_pia_mean:<18.2f}')
print(f'{"RMSE @ " + str(low_snr) + "dB (deg)":<32}{rmse_baseline[low_snr]:<18.3f}{rmse_pia[low_snr]:<18.3f}')
print(f'{"RMSE @ " + str(high_snr) + "dB (deg)":<32}{rmse_baseline[high_snr]:<18.3f}{rmse_pia[high_snr]:<18.3f}')
print(f'{"Pd @ " + str(low_snr) + "dB":<32}{pd_baseline[low_snr]:<18.3f}{pd_pia[low_snr]:<18.3f}')
print(f'{"Pd @ " + str(high_snr) + "dB":<32}{pd_baseline[high_snr]:<18.3f}{pd_pia[high_snr]:<18.3f}')

In [ ]:
lighter = 'PIA-Net' if pia_params < baseline_params else 'Baseline UNet'
faster = 'PIA-Net' if lat_pia_mean < lat_baseline_mean else 'Baseline UNet'
better_low_snr = 'PIA-Net' if rmse_pia[low_snr] < rmse_baseline[low_snr] else 'Baseline UNet'
better_high_snr = 'PIA-Net' if rmse_pia[high_snr] < rmse_baseline[high_snr] else 'Baseline UNet'

print('AUTO-GENERATED VERDICT (ei run-er real number theke, hardcoded na):\n')
print(f'- Lightweight (fewer params): {lighter}')
print(f'- Faster inference: {faster}')
print(f'- Better RMSE at low SNR ({low_snr}dB, hardest case): {better_low_snr}')
print(f'- Better RMSE at high SNR ({high_snr}dB, easiest case): {better_high_snr}')
print()
print('MONE RAKHO: PIA-Net matro', EPOCHS, 'epoch train hoyeche ei notebook-e (baseline\'s')
print('paper Table I-er full 500-epoch training peyeche). EPOCHS/STEPS_PER_EPOCH barhale')
print('PIA-Net-er result aro improve hobar kotha — eta ekta *first-look*, final verdict na.')

## Porborti step

- `EPOCHS`/`STEPS_PER_EPOCH` (Part 10) barhao paper-er full setup-er kache pouchte — real fair
  comparison tokhon-i pabe।
- `N_PER_CONDITION` (Part 12) `1000` kore dile paper-er exact scale-e evaluation hobe (beshi shomoy nebe)।
- Field Guide artifact-e (age deya) aro advanced version-er idea ache — differentiable set-prediction
  head diye blob-detector-take pura shorate — eta ei notebook-e nei, porer step hishebe try kora jete pare।